# Projeto da disciplina de Processamento de imagens

O projeto consiste em software de detecção de raças de cachorros, utilizando processamento de imagens e IA


In [ ]:
# Importando bibliotecas necessárias
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from ultralytics import YOLO
import cv2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from ultralytics import YOLO
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Dataset
dataset = "imagens"

racas = sorted(os.listdir(dataset))

print("Número de raças:", len(racas))

In [ ]:
# Contar o número de imagens por raça
contagem_imagens = {}

for raca in racas:
    pasta = os.path.join(dataset, raca)

    imagens = [
        arquivo
        for arquivo in os.listdir(pasta)
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    contagem_imagens[raca] = len(imagens)

    print(f"Raça: {raca}, Número de imagens: {len(imagens)}")

In [ ]:
plt.figure(figsize=(15, 5))

plt.bar(
    contagem_imagens.keys(),
    contagem_imagens.values()
)

plt.xticks(rotation=90)
plt.ylabel("Quantidade de imagens")
plt.title("Distribuição de imagens por raça")

plt.show()

In [ ]:
# Tamanho das imagens por raça

dataset_path = r"C:\Users\maria\OneDrive\Documentos\projetoPSRaças\imagens"

dados = []

for raiz, pastas, arquivos in os.walk(dataset_path):

    # Seleciona apenas arquivos de imagem
    imagens = [
        arquivo for arquivo in arquivos
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    # Só processa pastas que possuem imagens
    if imagens:

        tamanhos = []
        imagens_invalidas = 0

        for arquivo in imagens:

            caminho = os.path.join(raiz, arquivo)

            try:
                with Image.open(caminho) as imagem:
                    largura, altura = imagem.size
                    tamanhos.append((largura, altura))

            except:
                imagens_invalidas += 1

        if tamanhos:

            larguras = [t[0] for t in tamanhos]
            alturas = [t[1] for t in tamanhos]

            # Nome da pasta que contém as imagens
            raca = os.path.basename(raiz)

            dados.append({
                "Raça": raca,
                "Quantidade de imagens": len(tamanhos),

                "Largura mínima": min(larguras),
                "Largura máxima": max(larguras),
                "Largura média": round(sum(larguras) / len(larguras), 2),

                "Altura mínima": min(alturas),
                "Altura máxima": max(alturas),
                "Altura média": round(sum(alturas) / len(alturas), 2),

                "Imagens inválidas": imagens_invalidas
            })

df = pd.DataFrame(dados)

df = df.sort_values("Raça")

df

df.to_csv("analise_tamanhos_imagens_racas.csv", index=False, encoding="utf-8-sig")

In [ ]:
import os
import cv2
from ultralytics import YOLO

# 1. Carregar o modelo YOLOv8 pré-treinado
model = YOLO("yolov8n.pt")  # Pode usar o yolov8n.pt padrão para caixas delimitadoras

# 2. Pastas de entrada e saída
pasta_entrada = "images"
pasta_saida = "imagens_recortadas2"

total_processadas = 0
total_recortadas = 0
total_nao_detectadas = 0

# 3. Varredura com os.walk
for raiz, diretorios, arquivos in os.walk(pasta_entrada):
    for nome_arq in arquivos:
        if nome_arq.lower().endswith(('.png', '.jpg', '.jpeg')):
            caminho_in = os.path.join(raiz, nome_arq)
            total_processadas += 1

            # 3.1 Recria a subpasta da raça no diretório de saída
            caminho_relat = os.path.relpath(raiz, pasta_entrada)
            pasta_saida_atual = os.path.join(pasta_saida, caminho_relat)
            os.makedirs(pasta_saida_atual, exist_ok=True)
            caminho_out = os.path.join(pasta_saida_atual, nome_arq)

            # 4. Lê a imagem 
            img = cv2.imread(caminho_in)
            if img is None:
                continue

            # 5. O YOLO faz a detecção (confiança mínima de 0.4 para evitar falsos positivos)
            results = model(img, conf=0.4, verbose=False)

            cachorro_encontrado = False
            for result in results:
                if len(result.boxes) > 0:
                    for box in result.boxes:
                        # Classe 16 no COCO dataset é 'dog'
                        if int(box.cls[0]) == 16:
                            # 6. Extrai as coordenadas da caixa delimitadora
                            x1, y1, x2, y2 = map(int, box.xyxy[0])
                            
                            # Opcional: Adicionar uma margem de segurança pequena (padding) 
                            # para não colar muito nas bordas do cão
                            h, w, _ = img.shape
                            padding = 10
                            x1 = max(0, x1 - padding)
                            y1 = max(0, y1 - padding)
                            x2 = min(w, x2 + padding)
                            y2 = min(h, y2 + padding)

                            # 7. Faz o recorte da imagem usando fatiamento
                            recorte = img[y1:y2, x1:x2]

                            # 8. Salva a imagem recortada
                            if recorte.size > 0:
                                cv2.imwrite(caminho_out, recorte)
                                total_recortadas += 1
                                cachorro_encontrado = True
                                break
                    if cachorro_encontrado:
                        break
            
            if not cachorro_encontrado:
                total_nao_detectadas += 1
                # Opcional: Se o YOLO não achar o cachorro, copia a imagem original 
                # para não perder a amostra no treino
                cv2.imwrite(caminho_out, img)

print(f"\n--- Processamento Concluído ---")
print(f"Total de imagens lidas: {total_processadas}")
print(f"Imagens recortadas com sucesso pelo YOLO: {total_recortadas}")
print(f"Imagens onde o cão não foi detectado (mantidas originais): {total_nao_detectadas}")

In [ ]:
# Garantir a Qualidade e Limpeza da Base de Dados
import os
from PIL import Image

def limpar_imagens_corrompidas(diretorio_raiz):
    print("Inspecionando dataset em busca de arquivos corrompidos...")
    removidos = 0
    for pasta_raiz, _, arquivos in os.walk(diretorio_raiz):
        for arquivo in arquivos:
            caminho_arquivo = os.path.join(pasta_raiz, arquivo)
            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify() # Verifica se a imagem está íntegra
            except (IOError, SyntaxError):
                print(f"Removendo arquivo corrompido ou inválido: {caminho_arquivo}")
                os.remove(caminho_arquivo)
                removidos += 1
    print(f"Limpeza concluída! Total de arquivos corrompidos removidos: {removidos}")

# Exemplo de uso:
limpar_imagens_corrompidas("imagens_recortadas")

In [ ]:
# 1. Caminho absoluto da pasta raiz contendo as 120 subpastas de raças
pasta_treino = "imagens_recortadas"  # Ajuste se o caminho for diferente no seu PC

tamanho_imagem = (224, 224)
tamanho_lote = 32
epocas = 5  # Defina o número de épocas desejado

print("Configurando geradores com Data Augmentation e Validação (80/20)...")
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

gerador_treino = datagen.flow_from_directory(
    pasta_treino,
    target_size=tamanho_imagem,
    batch_size=tamanho_lote,
    class_mode='categorical',
    subset='training'
)

gerador_validacao = datagen.flow_from_directory(
    pasta_treino,
    target_size=tamanho_imagem,
    batch_size=tamanho_lote,
    class_mode='categorical',
    subset='validation'
)

num_classes = gerador_treino.num_classes
print(f"--> Total de raças (classes) detectadas pelo gerador: {num_classes}")

if num_classes != 120:
    print("AVISO: O gerador não encontrou 120 classes. Verifique o caminho da pasta!")
else:
    print("Arquitetura pronta para treinar com as 120 classes. Construindo modelo...")

    # 2. MobileNetV2 Transfer Learning
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    previsoes = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=previsoes)

    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])

    print("Iniciando o treinamento completo (isso vai demorar na CPU)...")
    historico = model.fit(
        gerador_treino,
        validation_data=gerador_validacao,
        epochs=epocas
    )

    # 3. Salvar o modelo atualizado sobrescrevendo o arquivo .h5
    model.save("modelo_classificador_caes.h5")
    print("Sucesso! Modelo treinado com 120 classes e salvo como 'modelo_classificador_caes.h5'.")

In [ ]:
# Aperfeiçoar o Pré-processamento com o YOLO e o Data Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configuração avançada de Data Augmentation para o treino
datagen_treino = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,          # Rotações moderadas (simula inclinações da cabeça)
    width_shift_range=0.15,     # Translação horizontal leve
    height_shift_range=0.15,    # Translação vertical leve
    shear_range=0.1,            # Cisalhamento para variar o ângulo de visão
    zoom_range=0.2,             # Zoom aleatório (simula o cão mais perto ou mais longe)
    horizontal_flip=True,       # Espelhamento horizontal (essencial para rostos)
    fill_mode='nearest',        # Preenchimento de pixels vazios gerados pelas rotações
    validation_split=0.2        # Divisão automática de treino/validação (80/20)
)

# Gerador para a validação (apenas normalização, sem distorções de aumento)
datagen_validacao = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2
)

# Aplicando nas suas pastas recortadas pelo YOLO:
pasta_treino = "imagens_recortadas"

gerador_treino = datagen_treino.flow_from_directory(
    pasta_treino,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

gerador_validacao = datagen_validacao.flow_from_directory(
    pasta_treino,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

In [ ]:

# 1. Carregar o modelo salvo
model = load_model("modelo_classificador_caes.h5")
print("Modelo carregado com sucesso!")

# 2. Configurar o gerador para os dados de teste/validação
# Usamos os mesmos parâmetros de normalização e apontamos para a mesma pasta
pasta_treino = "imagens_recortadas"

datagen_teste = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2  # Mantém a mesma proporção de divisão usada no treino
)

gerador_validacao = datagen_teste.flow_from_directory(
    pasta_treino,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False # Importante para manter a ordem correta na avaliação
)

# 3. Avaliar o modelo no conjunto de teste
perda, acuracia = model.evaluate(gerador_validacao)

print(f"Perda (Loss) no conjunto de teste: {perda:.4f}")
print(f"Acurácia (Accuracy) do modelo: {acuracia * 100:.2f}%")
print("Classes encontradas pelo Keras:", gerador_validacao.class_indices)
print(f"Total de classes detectadas: {gerador_validacao.num_classes}")

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# 1. Carregar o modelo treinado com as 120 classes
model = load_model("modelo_classificador_caes.h5")
print("Modelo carregado com sucesso para predição!")

# 2. Apontar para a mesma pasta de treino para recuperar os nomes das raças em ordem alfabética
pasta_treino = "imagens_recortadas"  # Ajuste para o caminho da sua pasta raiz com as 120 raças
classes = sorted(os.listdir(pasta_treino))

def prever_raca_cachorro(caminho_imagem):
    # 3. Carregar a imagem nova e redimensionar para o padrão do MobileNetV2 (224x224)
    img = image.load_img(caminho_imagem, target_size=(224, 224))
    
    # 4. Converter a imagem para um array de números
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0) # Adiciona a dimensão do lote (batch)
    
    # 5. Normalizar os pixels (mesmo procedimento do treino: dividir por 255.0)
    x = x / 255.0
    
    # 6. Realizar a predição
    previsoes = model.predict(x)
    
    # 7. Identificar a classe com maior probabilidade
    indice_classe_mais_provavel = np.argmax(previsoes[0])
    probabilidade = previsoes[0][indice_classe_mais_provavel] * 100
    
    raca_prevista = classes[indice_classe_mais_provavel]
    
    print(f"\n--- Resultado da Predição ---")
    print(f"Raça identificada: {raca_prevista}")
    print(f"Confiança da IA: {probabilidade:.2f}%")

# Exemplo de uso: substitua pelo caminho de uma foto de teste no seu computador
# Dica: use uma foto limpa ou já recortada (focada na cabeça/rosto do cão)
prever_raca_cachorro("lhasa.jpg")

In [ ]:
# 1. Carregar o modelo recém-treinado
model = load_model("modelo_classificador_caes.h5")

# 2. Descobrir quantas classes (saídas) o modelo espera
num_classes_modelo = model.output_shape[-1]
print(f"O modelo .h5 foi treinado para reconhecer: {num_classes_modelo} classes.")

# 3. Conferir se bate com a sua pasta de imagens recortadas
import os
pasta_treino = "imagens_recortadas" # Substitua pelo seu caminho real
pastas_encontradas = os.listdir(pasta_treino)
print(f"Total de pastas de raças encontradas no diretório: {len(pastas_encontradas)}")